# Module 37 — Harness Engineering: From Model Wrapper to Production Control Plane

**Colab goal:** build the same agent capability in four deliberate steps: **minimal → guarded → durable → replayable**, while keeping the model replaceable and the harness responsible for deterministic controls.

This notebook is designed to run with **Python standard library only** and deterministic fake providers/tools. No paid API key is required. Optional provider integrations can be added later without changing the harness contracts.

## The engineering thesis

`Agent = Model + Harness + Environment + Tools + State + Policy + Budget + Verification`

The model proposes. The harness validates, authorizes, budgets, executes through bounded tools, verifies outcomes, persists safe state, and records evidence for debugging.


## Where this module sits in the course

Module 36 made the agent loop explicit. Module 37 now makes the **runtime around the loop** reusable and governable. Module 38 will extend these contracts into long-running autonomy with leases, heartbeats, durable waiting states, and resume-after-failure semantics.

### Practice contract

**Predict → Build → Try → Break → Debug → Measure → Improve → Defend**


## Industry architecture

```text
                 USER / EVENT
                      │
                      ▼
              Task + Identity
                      │
                Policy Scope
                      │
                      ▼
             ┌─────────────────┐
             │     HARNESS     │
             │ context / state │
             │ budget / policy │
             │ recovery / trace│
             └───────┬─────────┘
                     │ proposal
                     ▼
                  MODEL
                     │
               action proposal
                     ▼
        schema + auth + budget gate
                     │
                     ▼
               TOOL GATEWAY
                     │
                     ▼
             ENVIRONMENT / API
                     │
                     ▼
        verify → commit state → trace
                     │
             checkpoint / replay
```

The same pattern applies to coding agents, SRE, SOC, customer support, procurement, finance operations, manufacturing, telecom, and enterprise research.


In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field, asdict
from decimal import Decimal
from typing import Any, Callable
import copy, json, time, uuid

print('Python harness lab ready')


# Level 0 — Establish the problem with a deliberately weak agent

### Industry scenario
A customer-support assistant must investigate an order and, when policy allows, issue a refund.

The weak architecture is:

```text
user → model → refund()
```

Before adding controls, predict the likely failure modes:

- wrong customer or tenant
- malformed amount
- unauthorized refund
- duplicate refund after retry
- model trusting hostile tool output
- no durable recovery after a crash

The purpose of the next levels is to eliminate these classes systematically.

In [ ]:
orders = {
    'O100': {'tenant':'acme','customer':'C1','total': 120.0, 'status':'delivered'},
    'O200': {'tenant':'globex','customer':'C9','total': 900.0, 'status':'delivered'},
}

def weak_refund(order_id: str, amount: float):
    order = orders[order_id]
    print(f'UNSAFE REFUND: tenant={order["tenant"]} order={order_id} amount={amount}')
    return {'status':'success','order_id':order_id,'amount':amount}

weak_refund('O100', 75.0)


### Break it

Try asking for a refund above the allowed automatic threshold or for an order in another tenant. There is no deterministic boundary to stop the request yet.

# Level 1 — Minimal reusable harness

Build only the contracts needed to separate model proposals from execution: task identity, structured action proposals, tool registry, schema validation, and a deterministic execution path.


In [ ]:
@dataclass
class TaskContext:
    task_id: str
    actor_id: str
    tenant_id: str
    objective: str
    risk_class: str = 'LOW'

@dataclass
class ActionProposal:
    tool: str
    args: dict[str, Any]

@dataclass
class ToolSpec:
    name: str
    required_permission: str
    risk: str
    validator: Callable[[dict[str, Any]], None]
    handler: Callable[[TaskContext, dict[str, Any]], dict[str, Any]]

class ToolGateway:
    def __init__(self):
        self.tools: dict[str, ToolSpec] = {}

    def register(self, spec: ToolSpec):
        self.tools[spec.name] = spec

    def call(self, ctx: TaskContext, proposal: ActionProposal, permissions: set[str]):
        if proposal.tool not in self.tools:
            raise ValueError('unknown_tool')
        spec = self.tools[proposal.tool]
        if spec.required_permission not in permissions:
            raise PermissionError('unauthorized_tool')
        spec.validator(proposal.args)
        return spec.handler(ctx, proposal.args)

def validate_lookup(args):
    if not isinstance(args.get('order_id'), str):
        raise TypeError('order_id must be a string')

def lookup_order(ctx, args):
    order = orders.get(args['order_id'])
    if order is None or order['tenant'] != ctx.tenant_id:
        return {'status':'not_found'}
    return {'status':'ok', 'order': copy.deepcopy(order)}

gateway = ToolGateway()
gateway.register(ToolSpec('lookup_order','orders:read','LOW',validate_lookup,lookup_order))

ctx = TaskContext(str(uuid.uuid4()), 'agent-user', 'acme', 'investigate order O100')
print(gateway.call(ctx, ActionProposal('lookup_order', {'order_id':'O100'}), {'orders:read'}))


## Measure Level 1

Record which proposals are rejected before execution. This is the first important harness metric: **invalid action containment rate**.

Target behavior: changing the model should not change the authorization contract.

# Level 2 — Guarded harness: context + policy + budget + verification

Now add the production controls that should not live exclusively inside the prompt.

```text
Task Context
    │
    ├── context allocator
    ├── model proposal
    ├── schema validation
    ├── authorization
    ├── policy decision
    ├── budget check
    ├── bounded tool
    └── independent verification
```


In [ ]:
@dataclass
class Budget:
    model_calls: int = 5
    tool_calls: int = 8
    cost_usd: float = 0.50

    def charge_model(self, cost=0.01):
        if self.model_calls <= 0 or self.cost_usd < cost:
            raise RuntimeError('budget_exhausted')
        self.model_calls -= 1
        self.cost_usd -= cost

    def charge_tool(self):
        if self.tool_calls <= 0:
            raise RuntimeError('tool_budget_exhausted')
        self.tool_calls -= 1

def refund_validator(args):
    if not isinstance(args.get('order_id'), str):
        raise TypeError('order_id must be a string')
    amount = args.get('amount')
    if not isinstance(amount, (int, float)) or amount <= 0:
        raise ValueError('amount must be positive')

def policy_refund(ctx: TaskContext, args: dict[str, Any], approval_id: str | None):
    amount = float(args['amount'])
    if amount <= 100:
        return {'allowed': True, 'reason':'auto_refund'}
    if approval_id:
        return {'allowed': True, 'reason':'approved'}
    return {'allowed': False, 'reason':'approval_required'}

refund_ledger = {}

def refund_handler(ctx, args):
    key = args['idempotency_key']
    if key in refund_ledger:
        return {'status':'already_applied', 'refund_id':refund_ledger[key]}
    refund_id = 'R-' + uuid.uuid4().hex[:8]
    refund_ledger[key] = refund_id
    return {'status':'applied', 'refund_id':refund_id, 'amount':args['amount']}

def verify_refund(result, expected_amount):
    return result.get('status') in {'applied','already_applied'} and float(result.get('amount', expected_amount)) == float(expected_amount)


## Guided build — exact-action approval

For high-risk mutations, approval should bind to the **specific action**, not merely to the broad goal.

Example approval binding:

```text
task_id       = task-2048
tool          = issue_refund
tenant        = acme
order_id      = O100
amount        = 125.00
action_version= refund-v1
```

An approval for “help customer” must not silently authorize a different order or a larger amount.

In [ ]:
ctx = TaskContext('task-2048','support-17','acme','resolve customer issue','HIGH')
budget = Budget()

def guarded_refund(ctx, args, permissions, approval_id=None):
    budget.charge_tool()
    refund_validator(args)
    if 'refund:create' not in permissions:
        raise PermissionError('refund_permission_denied')
    order = orders.get(args['order_id'])
    if not order or order['tenant'] != ctx.tenant_id:
        raise PermissionError('tenant_boundary_violation')
    decision = policy_refund(ctx, args, approval_id)
    if not decision['allowed']:
        return {'status':'blocked', 'reason':decision['reason']}
    result = refund_handler(ctx, args)
    if not verify_refund(result, args['amount']):
        raise RuntimeError('verification_failed')
    return result

proposal = {
    'order_id':'O100', 'amount':125.0,
    'idempotency_key':'task-2048:O100:refund-v1'
}
print(guarded_refund(ctx, proposal, {'refund:create'}))
print(guarded_refund(ctx, proposal, {'refund:create'}, approval_id='APR-1001'))


### Break challenge — malicious model proposal

Change the proposal to: `tenant_id='globex'`, `order_id='O200'`, or a different amount.

Expected result: the harness still derives tenant scope from `TaskContext` and refuses the cross-tenant operation. The model must not be able to rewrite identity or authorization context.

# Level 3 — Context engineering under a finite window

Context is an allocation problem. Critical controls and current state should not be displaced by irrelevant history.

### Example policy

```text
P0  identity / tenant / security constraints
P1  current state / verified evidence / tool contracts
P2  recent observations / relevant history
P3  summaries / optional background
```


In [ ]:
def allocate_context(items, budget_tokens=120):
    priority = {0:0, 1:1, 2:2, 3:3}
    ordered = sorted(items, key=lambda x: (priority[x['priority']], -x['value']))
    selected, used = [], 0
    for item in ordered:
        if used + item['tokens'] <= budget_tokens:
            selected.append(item)
            used += item['tokens']
    return selected, used

items = [
    {'name':'tenant-policy','priority':0,'tokens':20,'value':100},
    {'name':'current-state','priority':1,'tokens':30,'value':90},
    {'name':'verified-evidence','priority':1,'tokens':40,'value':80},
    {'name':'recent-history','priority':2,'tokens':50,'value':60},
    {'name':'old-chat','priority':3,'tokens':70,'value':20},
]
selected, used = allocate_context(items, 100)
print('selected=', [x['name'] for x in selected], 'tokens=', used)


### Measure

Compare naive concatenation against priority allocation on a synthetic two-hour SOC investigation. Track:

- critical-evidence retention
- context tokens
- model-call latency (synthetic)
- cost per successful task

The key metric is not “largest context.” It is **decision-relevant evidence retained per unit of context budget**.

# Level 4 — Durable state, checkpointing, and safe recovery

A checkpoint must represent enough semantics to recover safely. The most important field is often **side-effect status**, not the step number.

```text
checkpoint
   │
   ▼
inspect external effect
   │
 ┌─┴──────────┐
 │            │
known       unknown
 │            │
continue    reconcile
 │            │
 └──────┬─────┘
        ▼
verified state commit
```


In [ ]:
@dataclass
class RunState:
    task_id: str
    status: str = 'RUNNING'
    step: str = 'start'
    side_effect_status: str = 'NOT_STARTED'
    observations: list[dict[str, Any]] = field(default_factory=list)
    version: int = 0

    def checkpoint(self):
        return copy.deepcopy(asdict(self))

state = RunState('task-2048')
state.step = 'issue_refund'
state.side_effect_status = 'SENT_TO_PROVIDER_UNKNOWN_RESULT'
state.version += 1
checkpoint = state.checkpoint()
print(json.dumps(checkpoint, indent=2))


### Break challenge — crash after external mutation

Simulate this sequence:

1. Send a refund request.
2. Crash before writing local success state.
3. Restore checkpoint.
4. Attempt blind retry.

The correct design **reconciles external status first** and then uses the same idempotency key.


# Level 5 — Replayable traces

Replay is a debugging contract, not a promise of perfect determinism. Freeze model/tool fixtures where practical and record the inputs that matter.

### Minimum useful trace

`run_id, sequence, tenant, state_version, model_version, harness_version, tool, sanitized_args, result_class, policy_version, budget_before_after, verification, error`


In [ ]:
trace = []

def emit(event, **data):
    trace.append({'seq':len(trace)+1,'event':event,'time':time.time(), **data})

emit('TASK_START', task_id='task-2048', tenant='acme', harness_version='0.1')
emit('MODEL_PROPOSAL', tool='issue_refund', args={'order_id':'O100','amount':125.0})
emit('POLICY_DENY', reason='approval_required', policy_version='refund-policy-v3')
emit('APPROVAL_RECEIVED', approval_id='APR-1001')
emit('TOOL_RESULT', tool='issue_refund', result_class='applied')
emit('VERIFIED', check='ledger_match')

for event in trace:
    print(event)


### Replay experiment

Run the same fixture twice. Then change only the harness policy from `refund-policy-v3` to `refund-policy-v4`. The trajectory should expose exactly where behavior diverged.

This is the bridge from “we have logs” to **causal debugging**.

# Level 6 — Failure injection matrix

| Failure | Expected harness behavior | Metric |
|---|---|---|
| malformed model JSON | reject proposal / bounded repair | repair rate |
| wrong tenant | deny before tool execution | cross-tenant deny rate |
| unauthorized tool | fail closed | policy violation count |
| malicious tool output | classify as data | injected-instruction acceptance |
| timeout | bounded retry / circuit break | retry amplification |
| unknown mutation result | reconcile before retry | duplicate-effect rate |
| context overflow | preserve P0/P1 | critical-evidence retention |
| stale checkpoint | reject or migrate explicitly | restore failure rate |
| budget exhaustion | stop safely | budget-breach rate |
| cancellation | propagate and clean up | cancellation completion time |


# Level 7 — Industry elevation

Use the same harness against progressively higher-consequence workloads.

### 1. Coding agent
Read repository → edit → test automatically. Merge and production deployment require explicit gates. Workspace and network permissions are bounded.

### 2. SRE / platform engineering
Read metrics and logs automatically. Restarting a service may be approved by policy. Production configuration mutation requires stronger verification.

### 3. SOC
Correlate alerts and enrich incidents automatically. Isolation or destructive remediation requires high-risk authorization and exact-action approval.

### 4. Customer support
Read CRM/order data automatically. Refunds, credits, and account changes are risk-tiered and idempotent.

### 5. Procurement / finance operations
Gather invoices, contracts, supplier history, and policy evidence. Payment or commitment actions are bounded by amount, role, approval, and exact resource.

### 6. Manufacturing / telecom
Diagnostics can be broad and read-heavy. Physical or service-disrupting commands require stronger controls, environment validation, and post-action verification.


# Level 8 — Production hardening checklist

Before calling a harness production-ready, verify:

- [ ] provider can be swapped without rewriting governance
- [ ] context has explicit priority and budget
- [ ] model output cannot change tenant identity
- [ ] every tool has a typed schema
- [ ] tool access is authorization-checked
- [ ] tool results are treated as untrusted data
- [ ] policy and budgets are enforced outside the model
- [ ] high-risk actions use exact-action approval
- [ ] side effects have idempotency/reconciliation semantics
- [ ] state transitions are explicit
- [ ] checkpoints capture safe recovery semantics
- [ ] cancellation and deadlines propagate
- [ ] traces contain enough metadata for diagnosis without secrets
- [ ] replay fixtures exist for important incidents
- [ ] generated skills/extensions require validation before promotion
- [ ] failure paths are measurable

The repository’s Module 37 practice target explicitly emphasizes model adapter, context manager, tool gateway, policy, state, budget, verification, telemetry, replay, and policy coverage; this notebook turns those targets into a progressive runnable sequence.


# Final capstone — AegisAI Harness v1

Build a reusable runtime for a synthetic enterprise support or SRE workflow.

## Required progression

**Stage A:** minimal model adapter + typed tool gateway.  
**Stage B:** context allocator + policy + budgets + verification.  
**Stage C:** durable state + checkpoint + idempotent mutation.  
**Stage D:** structured trace + controlled replay + failure matrix.  
**Stage E:** apply the same harness to a second industry scenario without changing governance contracts.

## Gold-level defense

Take one failed run and explain:

1. what the model proposed;
2. what the harness accepted or rejected;
3. which policy applied;
4. how the budget changed;
5. what state was committed;
6. what verification evidence existed;
7. what happened during interruption;
8. why a duplicate side effect was prevented;
9. how the run can be replayed;
10. why the design fits the target industry.

**Mastery rule:** if replacing the model changes your security, authorization, budget, recovery, or audit behavior, the harness boundary is not engineered well enough yet.
